# Improved SegFormer notebook

This version reuses the `data_new` pipeline from the improved notebook, keeps the balanced `1500`-per-class augmentation target, and fine-tunes a pre-trained SegFormer for the `background`, `person`, `car`, and `dog` segmentation task.


In [ ]:
# Requirements:

# 1. Implement and create our own image segmentation model from scratch using PyTorch or TensorFlow. This will involve designing the architecture, defining the loss function, 
# and training the model on a dataset of images with corresponding segmentation masks.

# 2. Image segmentation task using semantic segmentation with a pre-trained model (In our case segformer) and fine-tuning it on our dataset.
# Transformers follow an encoder-decoder architecture, where the encoder processes the input image and turns them into vector representations using multi-head self-attention
# and position-wise feed-forward networks.
# Decoder takes these vector representations and generates the segmented output using masked self-attention, encoder-decoder attention and feed-forward layers.

# Multi-head self-attention lets every token (pixel/patch) look at every other token and decide how important they are. 
# It turns each input vector into Query, Key, and Value vectors (Q, K, V). Query - what am i looking for? Key - what do i have? Value - what do i output?
# Then attention scores are calculated by taking the dot product of the Query with all Keys, dividing by the square root of the dimension of the Key vectors, 
# and applying a softmax function to get weights. Weights are values between 0 and 1 that indicate how much attention each token should pay to every other token.
# One head isn't enough to capture all the relationships in the data, so we use multiple heads to allow the model to attend to different parts of the input simultaneously.
# "This pixel belongs to a person, this pixel belongs to a car, this pixel belongs to a dog" - the model can learn to focus on different features for each class.

# Chosen classes: 
# 1. Person 
# 2. Car 
# 3. Dog

# We will use the Hugging Face Transformers library to load a pre-trained SegFormer model and fine-tune it on our dataset.


# 1. Implementing our own image segmentation model from scratch using PyTorch:

In [ ]:
# 1. Creating a U-Net architecture for image segmentation:
# U-net is a convolutional neural network architecture that is widely used for image segmentation tasks, where each pixel in an image is classified into a specific category.
# The architecture consists of an encoder, which exctracts features from the input image using 3x3 convolutional layers followed by ReLU activation and max pooling.
# 1. The kernel extracts features, 2. ReLU adds non-linearity, 3. Max pooling reduces the spatial dimensions of the feature maps while retaining important information. 
# 3. Max pooling reduces from a 2x2 matrix to a single value, taking only the biggest value in the 2x2 area, which helps to reduce the computational load and capture 
# the most important features.
# Decoder upsamples the feature maps back to the original image size using transposed convolutional layers, which are also known as deconvolutional layers.
# Combines features from the encoder with the upsampled features in the decoder using skip connections, which helps to preserve spatial information and improve 
# segmentation accuracy.
# Skip connections - the output of one layer is added to the output of a deeper layer, bypassing intermediate layers. This helps mitigate gradient vanishing problems.
# It also allows the model to reuse earlier features, which allows fine details from early layers to be combined with high-level features.
# Output layer uses a softmax activation function to produce a probability distribution over the classes for each pixel, allowing the model to classify each pixel into
# one of the predefined classes.
# Most typical loss function for image segmentation is the cross-entropy loss, which measures the difference between the predicted class probabilities and the true class 
# labels for each pixel.

# Install Needed libraries

In [ ]:
# Install PyTorch
%pip install torch torchvision

# Install other necessary libraries
%pip install numpy matplotlib tqdm


# Setting up SegFormer


In [ ]:
# SegFormer-specific code is configured in the training cells below.
# This notebook keeps the improved `data_new` loading, augmentation preview,
# training curves, and presentation flow, but fine-tunes a pre-trained SegFormer
# instead of training a custom U-Net from scratch.


# Training the U-Net Model



In [ ]:
import os
from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Sequence, Tuple

import numpy as np
import torch
from PIL import Image
from torch.utils.data import ConcatDataset, DataLoader, Dataset, Subset
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

FAST_MODE = True
CACHE_DATA = True
TRAIN_TARGET_PER_CLASS = 1500
FINAL_TARGET_PER_CLASS = 1500
TEST_TOTAL = 100
VAL_FRACTION = 0.15
SEED = 42
CHECKPOINT_DIR = Path.cwd() / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["background", "person", "car", "dog"]
NUM_CLASSES = len(CLASS_NAMES)
ID_PERSON, ID_CAR, ID_DOG = 1, 2, 3
CLASS_COLOR_MAP = np.array(
    [
        [0, 0, 0],
        [255, 80, 80],
        [80, 180, 255],
        [80, 220, 120],
    ],
    dtype=np.uint8,
)

CPU_CORES = os.cpu_count() or 8
IMAGE_SIZE = (128, 128) if FAST_MODE else (256, 256)
BATCH_SIZE = 24 if device.type == "cuda" else 8
NUM_WORKERS = 2 if device.type == "cuda" else 0

if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass
else:
    torch.set_num_threads(min(8, CPU_CORES))
    try:
        torch.set_num_interop_threads(1)
    except RuntimeError:
        pass


def seed_everything(seed: int = 42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(SEED)


@dataclass(frozen=True)
class SegAugmentConfig:
    hflip_p: float = 0.5
    affine_p: float = 0.4
    degrees: float = 12.0
    translate: float = 0.08
    scale: Tuple[float, float] = (0.9, 1.12)
    brightness: Tuple[float, float] = (0.85, 1.15)
    contrast: Tuple[float, float] = (0.88, 1.18)
    saturation: Tuple[float, float] = (0.88, 1.18)
    sharpness: Tuple[float, float] = (0.9, 1.1)
    blur_p: float = 0.15
    blur_kernel_size: int = 5


class CachedBinarySegFolderDataset(Dataset):
    """Loads binary masks and maps foreground pixels to one semantic class id."""

    def __init__(
        self,
        images_dir: Path,
        masks_dir: Path,
        *,
        foreground_class_id: int,
        size: Tuple[int, int],
        cache: bool = True,
        image_exts: Sequence[str] = (".jpg", ".jpeg", ".png", ".webp"),
        mask_exts: Sequence[str] = (".png", ".jpg", ".jpeg", ".gif"),
        mask_prefixes: Sequence[str] = ("",),
        mask_suffixes: Sequence[str] = ("",),
    ):
        self.images_dir = Path(images_dir)
        self.masks_dir = Path(masks_dir)
        self.foreground_class_id = int(foreground_class_id)
        self.size = tuple(size)
        self.cache = bool(cache)
        self.image_exts = tuple(e.lower() for e in image_exts)
        self.mask_exts = tuple(e.lower() for e in mask_exts)
        if isinstance(mask_prefixes, str):
            mask_prefixes = (mask_prefixes,)
        if isinstance(mask_suffixes, str):
            mask_suffixes = (mask_suffixes,)
        self.mask_prefixes = tuple(str(p) for p in mask_prefixes)
        self.mask_suffixes = tuple(str(s) for s in mask_suffixes)

        images = [
            p
            for p in sorted(self.images_dir.iterdir())
            if p.is_file() and p.suffix.lower() in self.image_exts
        ]
        if not images:
            raise FileNotFoundError(f"No images found in {self.images_dir}")

        self.pairs = []
        for img_path in images:
            found = None
            for prefix in self.mask_prefixes:
                for suffix in self.mask_suffixes:
                    for ext in self.mask_exts:
                        candidate = self.masks_dir / f"{prefix}{img_path.stem}{suffix}{ext}"
                        if candidate.exists():
                            found = candidate
                            break
                    if found is not None:
                        break
                if found is not None:
                    break
            if found is not None:
                self.pairs.append((img_path, found))

        if not self.pairs:
            example = images[0].stem
            raise FileNotFoundError(
                f"No matching masks found in {self.masks_dir}. Tried names like "
                f"'{self.mask_prefixes[0]}{example}{self.mask_suffixes[0]}<ext>'."
            )

        self.samples = None
        if self.cache:
            self.samples = [self._load_pair(img_path, mask_path) for img_path, mask_path in self.pairs]

    def _load_pair(self, img_path: Path, mask_path: Path):
        img = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        img = TF.resize(img, self.size, interpolation=InterpolationMode.BILINEAR)
        mask = TF.resize(mask, self.size, interpolation=InterpolationMode.NEAREST)

        img_t = TF.to_tensor(img)
        mask_np = np.array(mask, dtype=np.uint8)
        out = np.zeros_like(mask_np, dtype=np.uint8)
        out[mask_np > 0] = np.uint8(self.foreground_class_id)
        mask_t = torch.from_numpy(out)
        return img_t.contiguous(), mask_t.contiguous()

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        if self.samples is None:
            img_t, mask_t = self._load_pair(*self.pairs[idx])
        else:
            img_t, mask_t = self.samples[idx]
        return img_t.clone(), mask_t.clone()


class AugmentedDataset(Dataset):
    def __init__(self, base: Dataset, cfg: Optional[SegAugmentConfig] = None):
        self.base = base
        self.cfg = cfg

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, mask = self.base[idx]
        if self.cfg is None:
            return img, mask

        if torch.rand(1).item() < self.cfg.hflip_p:
            img = torch.flip(img, dims=[2])
            mask = torch.flip(mask, dims=[1])

        if torch.rand(1).item() < self.cfg.affine_p:
            max_dx = self.cfg.translate * img.shape[2]
            max_dy = self.cfg.translate * img.shape[1]
            translate = [
                int(round(torch.empty(1).uniform_(-max_dx, max_dx).item())),
                int(round(torch.empty(1).uniform_(-max_dy, max_dy).item())),
            ]
            angle = float(torch.empty(1).uniform_(-self.cfg.degrees, self.cfg.degrees).item())
            scale = float(torch.empty(1).uniform_(*self.cfg.scale).item())

            img = TF.affine(
                img,
                angle=angle,
                translate=translate,
                scale=scale,
                shear=[0.0, 0.0],
                interpolation=InterpolationMode.BILINEAR,
                fill=0.0,
            )
            mask = TF.affine(
                mask.unsqueeze(0).float(),
                angle=angle,
                translate=translate,
                scale=scale,
                shear=[0.0, 0.0],
                interpolation=InterpolationMode.NEAREST,
                fill=0.0,
            ).squeeze(0).to(torch.uint8)

        b = float(torch.empty(1).uniform_(*self.cfg.brightness).item())
        c = float(torch.empty(1).uniform_(*self.cfg.contrast).item())
        s = float(torch.empty(1).uniform_(*self.cfg.saturation).item())
        sharp = float(torch.empty(1).uniform_(*self.cfg.sharpness).item())
        img = TF.adjust_brightness(img, b)
        img = TF.adjust_contrast(img, c)
        img = TF.adjust_saturation(img, s)
        img = TF.adjust_sharpness(img, sharp)

        if self.cfg.blur_p > 0 and torch.rand(1).item() < self.cfg.blur_p:
            img = TF.gaussian_blur(img, kernel_size=self.cfg.blur_kernel_size)

        return img.clamp(0.0, 1.0).contiguous(), mask.contiguous()


class OversampledDataset(Dataset):
    """Deterministic oversampling to a fixed size."""

    def __init__(self, base: Dataset, length: int, seed: int):
        self.base = base
        self.length = int(length)
        if len(base) == 0:
            raise ValueError("Base dataset is empty")
        g = torch.Generator().manual_seed(int(seed))
        self.indices = torch.randint(0, len(base), (self.length,), generator=g).tolist()

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        return self.base[self.indices[idx]]


def split_train_val_test(n: int, n_test: int, val_fraction: float, seed: int):
    if n_test >= n:
        raise ValueError(f"n_test must be < n. Got n_test={n_test}, n={n}")
    g = torch.Generator().manual_seed(int(seed))
    perm = torch.randperm(n, generator=g).tolist()
    test_idx = perm[:n_test]
    remaining = perm[n_test:]
    n_val = max(1, int(round(len(remaining) * val_fraction)))
    n_val = min(n_val, len(remaining) - 1)
    val_idx = remaining[:n_val]
    train_idx = remaining[n_val:]
    return train_idx, val_idx, test_idx


DATA_ROOT_CANDIDATES = [Path.cwd() / "data_new", Path.cwd() / "data"]
DATA_ROOT = next((path for path in DATA_ROOT_CANDIDATES if path.exists()), DATA_ROOT_CANDIDATES[0])
print("Using data root:", DATA_ROOT)

CAR_CFG = SegAugmentConfig(
    hflip_p=0.5,
    affine_p=0.3,
    degrees=8.0,
    translate=0.05,
    scale=(0.94, 1.08),
    brightness=(0.9, 1.1),
    contrast=(0.92, 1.12),
    saturation=(0.92, 1.08),
    sharpness=(0.95, 1.1),
    blur_p=0.1,
)
PERSON_DOG_CFG = SegAugmentConfig(
    hflip_p=0.5,
    affine_p=0.45,
    degrees=14.0,
    translate=0.08,
    scale=(0.9, 1.12),
    brightness=(0.84, 1.18),
    contrast=(0.88, 1.18),
    saturation=(0.88, 1.18),
    sharpness=(0.9, 1.12),
    blur_p=0.15,
)

car_base = CachedBinarySegFolderDataset(
    DATA_ROOT / "car" / "images",
    DATA_ROOT / "car" / "masks",
    foreground_class_id=ID_CAR,
    size=IMAGE_SIZE,
    cache=CACHE_DATA,
)
person_base = CachedBinarySegFolderDataset(
    DATA_ROOT / "person" / "images",
    DATA_ROOT / "person" / "masks",
    foreground_class_id=ID_PERSON,
    size=IMAGE_SIZE,
    cache=CACHE_DATA,
)
dog_base = CachedBinarySegFolderDataset(
    DATA_ROOT / "dog" / "images",
    DATA_ROOT / "dog" / "masks",
    foreground_class_id=ID_DOG,
    size=IMAGE_SIZE,
    cache=CACHE_DATA,
    mask_prefixes=("", "annotated_"),
)

base_sizes = {"car": len(car_base), "person": len(person_base), "dog": len(dog_base)}
print("Base sizes:", base_sizes)

test_base = TEST_TOTAL // 3
remainder = TEST_TOTAL - 3 * test_base
test_counts = {"car": test_base, "person": test_base, "dog": test_base}
for key in ("car", "person", "dog")[:remainder]:
    test_counts[key] += 1
for key in list(test_counts.keys()):
    test_counts[key] = min(test_counts[key], base_sizes[key] - 2)
print("Test counts per class:", test_counts, "total=", sum(test_counts.values()))

car_train_idx, car_val_idx, car_test_idx = split_train_val_test(base_sizes["car"], test_counts["car"], VAL_FRACTION, SEED)
person_train_idx, person_val_idx, person_test_idx = split_train_val_test(base_sizes["person"], test_counts["person"], VAL_FRACTION, SEED + 1)
dog_train_idx, dog_val_idx, dog_test_idx = split_train_val_test(base_sizes["dog"], test_counts["dog"], VAL_FRACTION, SEED + 2)

train_base_subsets = {
    "car": Subset(car_base, car_train_idx),
    "person": Subset(person_base, person_train_idx),
    "dog": Subset(dog_base, dog_train_idx),
}
train_augmented_sources = {
    "car": AugmentedDataset(train_base_subsets["car"], CAR_CFG),
    "person": AugmentedDataset(train_base_subsets["person"], PERSON_DOG_CFG),
    "dog": AugmentedDataset(train_base_subsets["dog"], PERSON_DOG_CFG),
}

car_train = OversampledDataset(train_augmented_sources["car"], TRAIN_TARGET_PER_CLASS, seed=SEED + 10)
person_train = OversampledDataset(train_augmented_sources["person"], TRAIN_TARGET_PER_CLASS, seed=SEED + 20)
dog_train = OversampledDataset(train_augmented_sources["dog"], TRAIN_TARGET_PER_CLASS, seed=SEED + 30)

train_ds = ConcatDataset([car_train, person_train, dog_train])
val_ds = ConcatDataset([
    Subset(car_base, car_val_idx),
    Subset(person_base, person_val_idx),
    Subset(dog_base, dog_val_idx),
])
test_ds = ConcatDataset([
    Subset(car_base, car_test_idx),
    Subset(person_base, person_test_idx),
    Subset(dog_base, dog_test_idx),
])


def tensor_to_image(img_t: torch.Tensor):
    return (img_t.detach().cpu().permute(1, 2, 0).numpy().clip(0.0, 1.0) * 255).astype(np.uint8)


def mask_to_rgb(mask):
    if isinstance(mask, torch.Tensor):
        mask = mask.detach().cpu().numpy()
    mask = np.asarray(mask, dtype=np.uint8)
    return CLASS_COLOR_MAP[mask]


def overlay_mask(img_t: torch.Tensor, mask, alpha: float = 0.4):
    image_np = tensor_to_image(img_t).astype(np.float32)
    mask_rgb = mask_to_rgb(mask).astype(np.float32)
    return ((1.0 - alpha) * image_np + alpha * mask_rgb).clip(0, 255).astype(np.uint8)


print("Train size:", len(train_ds), "(balanced with augmentation to target/class =", TRAIN_TARGET_PER_CLASS, ")")
print("Val size  :", len(val_ds), "(real images, no augmentation)")
print("Test size :", len(test_ds), "(unseen images, no augmentation)")
print("CPU threads:", torch.get_num_threads(), "| workers:", NUM_WORKERS)

loader_kwargs = dict(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
)
if NUM_WORKERS > 0:
    loader_kwargs["prefetch_factor"] = 2

train_loader = DataLoader(train_ds, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_ds, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_ds, shuffle=False, **loader_kwargs)

xb, yb = next(iter(train_loader))
print("train batch image shape:", xb.shape, xb.dtype)
print("train batch mask shape :", yb.shape, yb.dtype)
print("train mask classes     :", torch.unique(yb).tolist())


## Augmentation preview

These plots show one original training image per class together with several random augmentations. This is useful during the presentation because it makes the balancing and augmentation strategy visible.


In [ ]:
import matplotlib.pyplot as plt

AUGMENTATION_PREVIEW_COUNT = 3


def show_augmentation_preview(base_subsets, augmented_sources, preview_count: int = 3):
    class_order = ["person", "car", "dog"]
    fig, axes = plt.subplots(len(class_order), preview_count + 2, figsize=(4 * (preview_count + 2), 4 * len(class_order)))
    if len(class_order) == 1:
        axes = np.expand_dims(axes, axis=0)

    for row, class_name in enumerate(class_order):
        original_img, original_mask = base_subsets[class_name][0]

        axes[row, 0].imshow(tensor_to_image(original_img))
        axes[row, 0].set_title(f"{class_name}: original image")
        axes[row, 0].axis("off")

        axes[row, 1].imshow(overlay_mask(original_img, original_mask, alpha=0.45))
        axes[row, 1].set_title("original mask overlay")
        axes[row, 1].axis("off")

        for col in range(preview_count):
            aug_img, aug_mask = augmented_sources[class_name][0]
            axes[row, col + 2].imshow(overlay_mask(aug_img, aug_mask, alpha=0.45))
            axes[row, col + 2].set_title(f"augmented #{col + 1}")
            axes[row, col + 2].axis("off")

    plt.tight_layout()
    plt.show()


show_augmentation_preview(train_base_subsets, train_augmented_sources, preview_count=AUGMENTATION_PREVIEW_COUNT)


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor

USE_AMP = (device.type == "cuda")
PRETRAINED_SEGFORMER_MODEL = "nvidia/segformer-b0-finetuned-ade-512-512"
RESUME_FROM_LOCAL_SEGFORMER = False
SEGFORMER_LOCAL_SAVE_DIR = CHECKPOINT_DIR / "segformer_local_best"
SEGFORMER_FINAL_SAVE_DIR = CHECKPOINT_DIR / "segformer_final_presentation"
SEGFORMER_EPOCHS = 10
SEGFORMER_LR = 6e-5
SEGFORMER_WEIGHT_DECAY = 1e-4
SEGFORMER_BATCH_SIZE = BATCH_SIZE

id2label = {i: name for i, name in enumerate(CLASS_NAMES)}
label2id = {name: i for i, name in id2label.items()}

if RESUME_FROM_LOCAL_SEGFORMER and SEGFORMER_LOCAL_SAVE_DIR.exists():
    segformer_processor = SegformerImageProcessor.from_pretrained(SEGFORMER_LOCAL_SAVE_DIR)
    segformer_model = SegformerForSemanticSegmentation.from_pretrained(SEGFORMER_LOCAL_SAVE_DIR).to(device)
    print("Resuming SegFormer from:", SEGFORMER_LOCAL_SAVE_DIR)
else:
    segformer_processor = SegformerImageProcessor(
        do_resize=True,
        size={"height": IMAGE_SIZE[0], "width": IMAGE_SIZE[1]},
        do_reduce_labels=False,
    )
    segformer_model = SegformerForSemanticSegmentation.from_pretrained(
        PRETRAINED_SEGFORMER_MODEL,
        num_labels=NUM_CLASSES,
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True,
    ).to(device)
    print("Starting SegFormer from pretrained model:", PRETRAINED_SEGFORMER_MODEL)


def segformer_collate(batch):
    images, masks = zip(*batch)
    images_np = [tensor_to_image(img) for img in images]
    masks_np = [mask.cpu().numpy().astype(np.uint8) for mask in masks]
    return segformer_processor(
        images=images_np,
        segmentation_maps=masks_np,
        return_tensors="pt",
    )


segformer_loader_kwargs = dict(
    batch_size=SEGFORMER_BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
    collate_fn=segformer_collate,
)
if NUM_WORKERS > 0:
    segformer_loader_kwargs["prefetch_factor"] = 2

segformer_train_loader = DataLoader(train_ds, shuffle=True, **segformer_loader_kwargs)
segformer_val_loader = DataLoader(val_ds, shuffle=False, **segformer_loader_kwargs)
segformer_test_loader = DataLoader(test_ds, shuffle=False, **segformer_loader_kwargs)


@torch.no_grad()
def estimate_class_weights(dataset, num_classes: int):
    counts = torch.zeros(num_classes, dtype=torch.float64)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    for _, yb in loader:
        counts += torch.bincount(yb.view(-1), minlength=num_classes).to(torch.float64)
    weights = counts.sum() / (num_classes * counts.clamp_min(1.0))
    weights = weights / weights.mean()
    return weights.to(device=device, dtype=torch.float32)


class SegmentationLoss(nn.Module):
    def __init__(self, ce_weight: torch.Tensor, dice_weight: float = 0.5):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=ce_weight)
        self.dice_weight = float(dice_weight)

    def forward(self, logits: torch.Tensor, targets: torch.Tensor):
        ce = self.ce(logits, targets)
        probs = torch.softmax(logits, dim=1)
        target_1h = F.one_hot(targets, num_classes=logits.shape[1]).permute(0, 3, 1, 2).float()

        probs_fg = probs[:, 1:]
        target_fg = target_1h[:, 1:]
        dims = (0, 2, 3)
        intersection = (probs_fg * target_fg).sum(dim=dims)
        denominator = probs_fg.sum(dim=dims) + target_fg.sum(dim=dims)
        dice = (2.0 * intersection + 1e-6) / (denominator + 1e-6)
        dice_loss = 1.0 - dice.mean()
        return ce + self.dice_weight * dice_loss


segformer_optimizer = torch.optim.AdamW(
    segformer_model.parameters(),
    lr=SEGFORMER_LR,
    weight_decay=SEGFORMER_WEIGHT_DECAY,
)
segformer_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    segformer_optimizer,
    mode="max",
    factor=0.5,
    patience=2,
)
segformer_class_weights = estimate_class_weights(train_ds, NUM_CLASSES)
segformer_criterion = SegmentationLoss(segformer_class_weights, dice_weight=0.5)
segformer_scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)


@torch.no_grad()
def evaluate_segformer(model, loader, num_classes: int):
    model.eval()
    total_loss = 0.0
    conf = torch.zeros((num_classes, num_classes), device=device, dtype=torch.int64)

    for batch in loader:
        pixel_values = batch["pixel_values"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, dtype=torch.long, non_blocking=True)

        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            outputs = model(pixel_values=pixel_values)
            logits = F.interpolate(
                outputs.logits,
                size=labels.shape[-2:],
                mode="bilinear",
                align_corners=False,
            )
            loss = segformer_criterion(logits, labels)

        total_loss += loss.item()
        preds = logits.argmax(dim=1)
        idx = NUM_CLASSES * labels.view(-1) + preds.view(-1)
        conf += torch.bincount(idx, minlength=num_classes * num_classes).reshape(num_classes, num_classes)

    conf = conf.float()
    diag = conf.diag()
    recall = diag / (conf.sum(1) + 1e-7)
    precision = diag / (conf.sum(0) + 1e-7)
    f1 = 2.0 * precision * recall / (precision + recall + 1e-7)
    denom = conf.sum(1) + conf.sum(0) - diag + 1e-7
    iou = diag / denom
    pixel_acc = (diag.sum() / (conf.sum() + 1e-7)).item()

    return {
        "loss": total_loss / max(1, len(loader)),
        "pixel_acc": pixel_acc,
        "miou_all": float(iou.mean().item()),
        "miou_fg": float(iou[1:].mean().item()),
        "iou_per_class": iou.detach().cpu().numpy(),
        "precision_per_class": precision.detach().cpu().numpy(),
        "recall_per_class": recall.detach().cpu().numpy(),
        "f1_per_class": f1.detach().cpu().numpy(),
    }


def train_one_epoch_segformer(model, loader):
    model.train()
    running_loss = 0.0

    for batch in tqdm(loader, desc="train-segformer", leave=False):
        pixel_values = batch["pixel_values"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, dtype=torch.long, non_blocking=True)

        segformer_optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            outputs = model(pixel_values=pixel_values)
            logits = F.interpolate(
                outputs.logits,
                size=labels.shape[-2:],
                mode="bilinear",
                align_corners=False,
            )
            loss = segformer_criterion(logits, labels)

        if USE_AMP:
            segformer_scaler.scale(loss).backward()
            segformer_scaler.step(segformer_optimizer)
            segformer_scaler.update()
        else:
            loss.backward()
            segformer_optimizer.step()

        running_loss += loss.item()

    return running_loss / max(1, len(loader))


best_val_miou = -float("inf")
best_epoch = 0
best_state = None
epochs_without_improvement = 0
history = []

print("SegFormer image size:", IMAGE_SIZE, "| batch size:", SEGFORMER_BATCH_SIZE, "| AMP:", USE_AMP)

for epoch in range(1, SEGFORMER_EPOCHS + 1):
    train_loss = train_one_epoch_segformer(segformer_model, segformer_train_loader)
    val_metrics = evaluate_segformer(segformer_model, segformer_val_loader, NUM_CLASSES)
    segformer_scheduler.step(val_metrics["miou_fg"])
    current_lr = segformer_optimizer.param_groups[0]["lr"]

    if val_metrics["miou_fg"] > best_val_miou:
        best_val_miou = val_metrics["miou_fg"]
        best_epoch = epoch
        best_state = {k: v.detach().cpu().clone() for k, v in segformer_model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    history.append(
        {
            "epoch": epoch,
            "train_loss": float(train_loss),
            "val_loss": float(val_metrics["loss"]),
            "pixel_acc": float(val_metrics["pixel_acc"]),
            "miou_fg": float(val_metrics["miou_fg"]),
            "lr": float(current_lr),
        }
    )

    print(
        f"Epoch {epoch:02d}/{SEGFORMER_EPOCHS} | "
        f"lr={current_lr:.2e} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_metrics['loss']:.4f} | "
        f"pixAcc={val_metrics['pixel_acc']:.3f} | "
        f"mIoU_fg={val_metrics['miou_fg']:.3f}"
    )

if best_state is not None:
    segformer_model.load_state_dict(best_state)

SEGFORMER_LOCAL_SAVE_DIR.mkdir(parents=True, exist_ok=True)
segformer_model.save_pretrained(SEGFORMER_LOCAL_SAVE_DIR)
segformer_processor.save_pretrained(SEGFORMER_LOCAL_SAVE_DIR)
print("Saved fine-tuned SegFormer to:", SEGFORMER_LOCAL_SAVE_DIR)
print("Best epoch:", best_epoch)

test_metrics = evaluate_segformer(segformer_model, segformer_test_loader, NUM_CLASSES)
print(
    f"SEGFORMER TEST | loss={test_metrics['loss']:.4f} | "
    f"pixAcc={test_metrics['pixel_acc']:.3f} | "
    f"mIoU_fg={test_metrics['miou_fg']:.3f}"
)
for idx, class_name in enumerate(CLASS_NAMES):
    print(
        f"{class_name:>10} | "
        f"IoU={test_metrics['iou_per_class'][idx]:.3f} | "
        f"Precision={test_metrics['precision_per_class'][idx]:.3f} | "
        f"Recall={test_metrics['recall_per_class'][idx]:.3f} | "
        f"F1={test_metrics['f1_per_class'][idx]:.3f}"
    )


## Training curves

After training, the plots below summarize how loss, validation pixel accuracy, validation foreground mIoU, and learning rate changed across epochs.


In [ ]:
import matplotlib.pyplot as plt

if not history:
    print("Run the training cell first to collect history.")
else:
    epochs = [row["epoch"] for row in history]
    train_losses = [row["train_loss"] for row in history]
    val_losses = [row["val_loss"] for row in history]
    pixel_accs = [row["pixel_acc"] for row in history]
    miou_fgs = [row["miou_fg"] for row in history]
    learning_rates = [row["lr"] for row in history]

    fig, axes = plt.subplots(1, 4, figsize=(24, 4.5))

    axes[0].plot(epochs, train_losses, marker="o", label="train loss")
    axes[0].plot(epochs, val_losses, marker="o", label="val loss")
    axes[0].set_title("Loss curves")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(epochs, pixel_accs, marker="o", color="tab:green")
    axes[1].set_title("Validation pixel accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].grid(alpha=0.3)

    axes[2].plot(epochs, miou_fgs, marker="o", color="tab:orange")
    axes[2].set_title("Validation foreground mIoU")
    axes[2].set_xlabel("Epoch")
    axes[2].grid(alpha=0.3)

    axes[3].plot(epochs, learning_rates, marker="o", color="tab:red")
    axes[3].set_title("Learning rate")
    axes[3].set_xlabel("Epoch")
    axes[3].set_yscale("log")
    axes[3].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()


## External Open Images evaluation

This section is for the **final external test** on 100 unseen Open Images images containing at least one of the classes: `Person`, `Car`, or `Dog`.


In [ ]:
# External evaluation on 100 unseen Open Images validation images
# %pip install fiftyone

import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode

import fiftyone.zoo as foz
from fiftyone.utils.labels import objects_to_segmentations

OI_CLASSES = ["Person", "Car", "Dog"]
OI_MASK_TARGETS = {1: "Person", 2: "Car", 3: "Dog"}
OI_MAX_SAMPLES = 100

oi_dataset = foz.load_zoo_dataset(
    "open-images-v7",
    split="validation",
    label_types=["segmentations"],
    classes=OI_CLASSES,
    only_matching=True,
    max_samples=OI_MAX_SAMPLES,
    shuffle=True,
    seed=SEED,
    dataset_name="open-images-v7-person-car-dog-seg-100",
)

instance_field = None
for candidate in ("ground_truth", "detections", "segmentations"):
    try:
        objects_to_segmentations(
            oi_dataset,
            candidate,
            "gt_semantic",
            mask_targets=OI_MASK_TARGETS,
            overwrite=True,
            progress=True,
        )
        instance_field = candidate
        break
    except Exception:
        continue

if instance_field is None:
    raise RuntimeError(
        "Could not find the Open Images instance-segmentation field. "
        "Inspect `oi_dataset.get_field_schema()` and update the candidate list."
    )

print("Open Images masks converted from field:", instance_field)
print("Downloaded samples:", len(oi_dataset))


class OpenImagesSemanticDataset(Dataset):
    def __init__(self, samples, size):
        self.samples = list(samples.iter_samples(progress=True))
        self.size = tuple(size)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        img = Image.open(sample.filepath).convert("RGB")
        img = TF.resize(img, self.size, interpolation=InterpolationMode.BILINEAR)
        img_t = TF.to_tensor(img)
        mask_np = sample["gt_semantic"].get_mask()
        if mask_np is None:
            raise RuntimeError(f"No semantic mask found for sample: {sample.filepath}")
        mask_t = torch.from_numpy(mask_np.astype("uint8"))
        mask_t = TF.resize(
            mask_t.unsqueeze(0).float(),
            self.size,
            interpolation=InterpolationMode.NEAREST,
        ).squeeze(0).to(torch.long)
        return img_t.contiguous(), mask_t.contiguous()


oi_ds = OpenImagesSemanticDataset(oi_dataset, IMAGE_SIZE)
oi_loader = DataLoader(
    oi_ds,
    batch_size=SEGFORMER_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
    collate_fn=segformer_collate,
)

oi_metrics = evaluate_segformer(segformer_model, oi_loader, NUM_CLASSES)
print(
    f"OPENIMAGES-100 | loss={oi_metrics['loss']:.4f} | "
    f"pixAcc={oi_metrics['pixel_acc']:.3f} | "
    f"mIoU_fg={oi_metrics['miou_fg']:.3f}"
)

for idx, class_name in enumerate(CLASS_NAMES):
    print(
        f"{class_name:>10} | "
        f"IoU={oi_metrics['iou_per_class'][idx]:.3f} | "
        f"Precision={oi_metrics['precision_per_class'][idx]:.3f} | "
        f"Recall={oi_metrics['recall_per_class'][idx]:.3f} | "
        f"F1={oi_metrics['f1_per_class'][idx]:.3f}"
    )


## Final training on all local data + external Open Images test

This will:
1. Rebuild a training set from **the local dataset**
2. Retrain a fresh model for the selected number of epochs
3. Download and evaluate on **100 unseen Open Images validation images**


In [ ]:
# Final model: fine-tune SegFormer on all local data, then evaluate on unseen Open Images images

FINAL_SEGFORMER_EPOCHS = best_epoch if "best_epoch" in globals() and best_epoch > 0 else SEGFORMER_EPOCHS
print("Final SegFormer training epochs:", FINAL_SEGFORMER_EPOCHS)

full_car_train = OversampledDataset(AugmentedDataset(car_base, CAR_CFG), FINAL_TARGET_PER_CLASS, seed=SEED + 110)
full_person_train = OversampledDataset(AugmentedDataset(person_base, PERSON_DOG_CFG), FINAL_TARGET_PER_CLASS, seed=SEED + 120)
full_dog_train = OversampledDataset(AugmentedDataset(dog_base, PERSON_DOG_CFG), FINAL_TARGET_PER_CLASS, seed=SEED + 130)
final_train_ds = ConcatDataset([full_car_train, full_person_train, full_dog_train])

final_segformer_loader = DataLoader(
    final_train_ds,
    shuffle=True,
    batch_size=SEGFORMER_BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
    collate_fn=segformer_collate,
    **({"prefetch_factor": 2} if NUM_WORKERS > 0 else {}),
)

if SEGFORMER_LOCAL_SAVE_DIR.exists():
    final_model = SegformerForSemanticSegmentation.from_pretrained(SEGFORMER_LOCAL_SAVE_DIR).to(device)
    print("Starting final fine-tuning from local best SegFormer:", SEGFORMER_LOCAL_SAVE_DIR)
else:
    final_model = SegformerForSemanticSegmentation.from_pretrained(
        PRETRAINED_SEGFORMER_MODEL,
        num_labels=NUM_CLASSES,
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True,
    ).to(device)
    print("Local best SegFormer not found, starting from pretrained model.")

final_optimizer = torch.optim.AdamW(final_model.parameters(), lr=SEGFORMER_LR, weight_decay=SEGFORMER_WEIGHT_DECAY)
final_class_weights = estimate_class_weights(final_train_ds, NUM_CLASSES)
final_criterion = SegmentationLoss(final_class_weights, dice_weight=0.5)
final_scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)


def train_one_epoch_final_segformer(model, loader):
    model.train()
    running_loss = 0.0

    for batch in tqdm(loader, desc="final-train-segformer", leave=False):
        pixel_values = batch["pixel_values"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, dtype=torch.long, non_blocking=True)

        final_optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            outputs = model(pixel_values=pixel_values)
            logits = F.interpolate(
                outputs.logits,
                size=labels.shape[-2:],
                mode="bilinear",
                align_corners=False,
            )
            loss = final_criterion(logits, labels)

        if USE_AMP:
            final_scaler.scale(loss).backward()
            final_scaler.step(final_optimizer)
            final_scaler.update()
        else:
            loss.backward()
            final_optimizer.step()

        running_loss += loss.item()

    return running_loss / max(1, len(loader))


for epoch in range(1, FINAL_SEGFORMER_EPOCHS + 1):
    final_train_loss = train_one_epoch_final_segformer(final_model, final_segformer_loader)
    print(f"Final epoch {epoch:02d}/{FINAL_SEGFORMER_EPOCHS} | train_loss={final_train_loss:.4f}")

SEGFORMER_FINAL_SAVE_DIR.mkdir(parents=True, exist_ok=True)
final_model.save_pretrained(SEGFORMER_FINAL_SAVE_DIR)
segformer_processor.save_pretrained(SEGFORMER_FINAL_SAVE_DIR)
print("Saved final SegFormer checkpoint to:", SEGFORMER_FINAL_SAVE_DIR)

if "oi_loader" not in globals():
    import fiftyone.zoo as foz
    from fiftyone.utils.labels import objects_to_segmentations

    OI_CLASSES = ["Person", "Car", "Dog"]
    OI_MASK_TARGETS = {1: "Person", 2: "Car", 3: "Dog"}
    OI_MAX_SAMPLES = 100

    oi_dataset = foz.load_zoo_dataset(
        "open-images-v7",
        split="validation",
        label_types=["segmentations"],
        classes=OI_CLASSES,
        only_matching=True,
        max_samples=OI_MAX_SAMPLES,
        shuffle=True,
        seed=SEED,
        dataset_name="open-images-v7-person-car-dog-seg-100",
    )

    instance_field = None
    for candidate in ("ground_truth", "detections", "segmentations"):
        try:
            objects_to_segmentations(
                oi_dataset,
                candidate,
                "gt_semantic",
                mask_targets=OI_MASK_TARGETS,
                overwrite=True,
                progress=True,
            )
            instance_field = candidate
            break
        except Exception:
            continue

    if instance_field is None:
        raise RuntimeError(
            "Could not find the Open Images instance-segmentation field. "
            "Inspect `oi_dataset.get_field_schema()` and update the candidate list."
        )

    class OpenImagesSemanticDataset(Dataset):
        def __init__(self, samples, size):
            self.samples = list(samples.iter_samples(progress=True))
            self.size = tuple(size)

        def __len__(self):
            return len(self.samples)

        def __getitem__(self, idx):
            sample = self.samples[idx]
            img = Image.open(sample.filepath).convert("RGB")
            img = TF.resize(img, self.size, interpolation=InterpolationMode.BILINEAR)
            img_t = TF.to_tensor(img)
            mask_np = sample["gt_semantic"].get_mask()
            if mask_np is None:
                raise RuntimeError(f"No semantic mask found for sample: {sample.filepath}")
            mask_t = torch.from_numpy(mask_np.astype("uint8"))
            mask_t = TF.resize(
                mask_t.unsqueeze(0).float(),
                self.size,
                interpolation=InterpolationMode.NEAREST,
            ).squeeze(0).to(torch.long)
            return img_t.contiguous(), mask_t.contiguous()

    oi_ds = OpenImagesSemanticDataset(oi_dataset, IMAGE_SIZE)
    oi_loader = DataLoader(
        oi_ds,
        batch_size=SEGFORMER_BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=(device.type == "cuda"),
        collate_fn=segformer_collate,
    )

original_criterion = segformer_criterion
segformer_criterion = final_criterion
oi_metrics = evaluate_segformer(final_model, oi_loader, NUM_CLASSES)
segformer_criterion = original_criterion

print(
    f"FINAL OPENIMAGES-100 | loss={oi_metrics['loss']:.4f} | "
    f"pixAcc={oi_metrics['pixel_acc']:.3f} | "
    f"mIoU_fg={oi_metrics['miou_fg']:.3f}"
)
for idx, class_name in enumerate(CLASS_NAMES):
    print(
        f"{class_name:>10} | "
        f"IoU={oi_metrics['iou_per_class'][idx]:.3f} | "
        f"Precision={oi_metrics['precision_per_class'][idx]:.3f} | "
        f"Recall={oi_metrics['recall_per_class'][idx]:.3f} | "
        f"F1={oi_metrics['f1_per_class'][idx]:.3f}"
    )


## Load saved weights and predict on new images

To use during the presentation for the selected images. Load the saved checkpoint, and visualize the predicted mask.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor

MODEL_DIR_TO_LOAD = CHECKPOINT_DIR / "segformer_final_presentation"
FALLBACK_MODEL_DIR = CHECKPOINT_DIR / "segformer_local_best"
if not MODEL_DIR_TO_LOAD.exists() and FALLBACK_MODEL_DIR.exists():
    MODEL_DIR_TO_LOAD = FALLBACK_MODEL_DIR

PRESENTATION_IMAGE_PATHS = []
# PRESENTATION_IMAGE_PATHS = [Path(r"C:\path\to\image1.jpg"), Path(r"C:\path\to\image2.jpg")]

inference_processor = SegformerImageProcessor.from_pretrained(MODEL_DIR_TO_LOAD)
inference_model = SegformerForSemanticSegmentation.from_pretrained(MODEL_DIR_TO_LOAD).to(device)
inference_model.eval()
print("Loaded SegFormer from:", MODEL_DIR_TO_LOAD)

color_map = CLASS_COLOR_MAP


@torch.no_grad()
def show_segformer_prediction(image_path: Path):
    img = Image.open(image_path).convert("RGB")
    original_size = img.size
    encoded = inference_processor(images=np.array(img), return_tensors="pt")
    pixel_values = encoded["pixel_values"].to(device)

    outputs = inference_model(pixel_values=pixel_values)
    logits = F.interpolate(
        outputs.logits,
        size=(original_size[1], original_size[0]),
        mode="bilinear",
        align_corners=False,
    )
    pred = logits.argmax(dim=1).squeeze(0).cpu().numpy().astype(np.uint8)

    pred_rgb = color_map[pred]
    overlay = (0.6 * np.array(img).astype(np.float32) + 0.4 * pred_rgb.astype(np.float32)).clip(0, 255).astype(np.uint8)

    fig, ax = plt.subplots(1, 3, figsize=(14, 5))
    ax[0].imshow(img)
    ax[0].set_title("Image")
    ax[0].axis("off")

    ax[1].imshow(pred, vmin=0, vmax=len(CLASS_NAMES) - 1, cmap="tab10")
    ax[1].set_title("Predicted mask")
    ax[1].axis("off")

    ax[2].imshow(overlay)
    ax[2].set_title("Overlay")
    ax[2].axis("off")

    plt.tight_layout()
    plt.show()
    print("Predicted classes:", [CLASS_NAMES[i] for i in np.unique(pred)])


if not PRESENTATION_IMAGE_PATHS:
    print("Set PRESENTATION_IMAGE_PATHS before running this cell.")
else:
    for image_path in PRESENTATION_IMAGE_PATHS:
        print("Running inference on:", image_path)
        show_segformer_prediction(Path(image_path))


## Qualitative local test predictions

This section visualizes how the trained model classifies pixels on unseen local test images. For each example it shows the input image, the ground-truth mask, the predicted mask, an overlay, probability heatmaps for `person`, `car`, and `dog`, and a correctness map.


In [ ]:
import matplotlib.pyplot as plt

QUALITATIVE_NUM_SAMPLES = 3
QUALITATIVE_SEED = SEED
qualitative_model = final_model if "final_model" in globals() else inference_model if "inference_model" in globals() else segformer_model
qualitative_model.eval()


@torch.no_grad()
def predict_segformer_sample_with_probs(model, image_t: torch.Tensor):
    encoded = segformer_processor(images=tensor_to_image(image_t), return_tensors="pt")
    pixel_values = encoded["pixel_values"].to(device)
    outputs = model(pixel_values=pixel_values)
    logits = F.interpolate(
        outputs.logits,
        size=image_t.shape[-2:],
        mode="bilinear",
        align_corners=False,
    )
    probs = torch.softmax(logits, dim=1).squeeze(0).cpu()
    pred = probs.argmax(dim=0).to(torch.uint8)
    return probs, pred


def show_segformer_qualitative_predictions(model, dataset, num_samples: int = 3, seed: int = 42):
    if len(dataset) == 0:
        print("Dataset is empty.")
        return

    rng = np.random.default_rng(seed)
    indices = rng.choice(len(dataset), size=min(num_samples, len(dataset)), replace=False)

    for sample_idx in indices:
        image_t, target_t = dataset[int(sample_idx)]
        probs, pred = predict_segformer_sample_with_probs(model, image_t)
        overlay = overlay_mask(image_t, pred, alpha=0.45)
        correctness = (pred == target_t).numpy().astype(np.float32)
        sample_pixel_acc = float((pred == target_t).float().mean().item())

        fig, axes = plt.subplots(2, 4, figsize=(18, 9))
        axes[0, 0].imshow(tensor_to_image(image_t))
        axes[0, 0].set_title("Input image")
        axes[0, 0].axis("off")

        axes[0, 1].imshow(mask_to_rgb(target_t))
        axes[0, 1].set_title("Ground-truth mask")
        axes[0, 1].axis("off")

        axes[0, 2].imshow(mask_to_rgb(pred))
        axes[0, 2].set_title("Predicted mask")
        axes[0, 2].axis("off")

        axes[0, 3].imshow(overlay)
        axes[0, 3].set_title("Prediction overlay")
        axes[0, 3].axis("off")

        for ax, class_idx in zip(axes[1, :3], [ID_PERSON, ID_CAR, ID_DOG]):
            heatmap = probs[class_idx].numpy()
            im = ax.imshow(heatmap, cmap="magma", vmin=0.0, vmax=1.0)
            ax.set_title(f"P({CLASS_NAMES[class_idx]})")
            ax.axis("off")
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

        axes[1, 3].imshow(correctness, cmap="RdYlGn", vmin=0.0, vmax=1.0)
        axes[1, 3].set_title("Correctness map")
        axes[1, 3].axis("off")

        fig.suptitle(f"Local test sample {int(sample_idx)} | pixel accuracy = {sample_pixel_acc:.3f}", fontsize=14)
        plt.tight_layout()
        plt.show()

        present_classes = [CLASS_NAMES[i] for i in torch.unique(pred).tolist()]
        print("Predicted classes:", present_classes)
        print()


show_segformer_qualitative_predictions(qualitative_model, test_ds, num_samples=QUALITATIVE_NUM_SAMPLES, seed=QUALITATIVE_SEED)
